# Combined CMiO and North Batten sensitivity: Results figure and supplementary table

Run All to call the **existing CMiO and North Batten sensitivity workflows**, then generate one compact **2 x 2 Results figure** and a full **mean ± sample SD supplementary table**. The three individual sensitivity notebooks remain the experiment definitions.

Both datasets use the same 30 algorithm seeds (0-29), fitting and evaluation neighbourhoods $k=10$, and reference class counts $K=7$ (CMiO) and $K=15$ (North Batten). The overlap sweep varies $f \in \{0,0.25,0.5,0.75,1\}$ with $R=25$; the ramp sweep varies $R \in \{0,10,25,50,100\}$ with $f=0.5$. These are separate one-parameter sweeps, not an $f \times R$ grid. The shared $(f,R)=(0.5,25)$ baseline is fitted once per seed and dataset.

Run All performs **540 distinct fits** (270 per dataset), reusing the shared baseline within each dataset. Scores and summary tables stay in memory, and the figure and table are displayed directly in this notebook. No data, figures or captions are exported.


In [1]:
from __future__ import annotations
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import HTML, display
from matplotlib_inline.backend_inline import set_matplotlib_formats

set_matplotlib_formats('retina')

def find_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        for folder in (candidate, candidate / 'sc-pacmap-reproducibility'):
            if (folder / 'code' / 'sc_pacmap' / 'sc_pacmap.py').is_file():
                return folder
    raise FileNotFoundError('Run from within the project directory.')

ROOT = find_root()
SOURCE_DIR = ROOT / 'notebooks' / 'sensitivity test'
DATASETS = {
    'cmio': ('CMiO', '01_cmio_sc_pacmap_sensitivity.ipynb', 7),
    'north_batten': ('North Batten', '02_north_batten_sc_pacmap_sensitivity.ipynb', 15),
}
SEEDS = tuple(range(30))
OVERLAP_VALUES = (0.0, 0.25, 0.5, 0.75, 1.0)
RAMP_VALUES = (0, 10, 25, 50, 100)
# Setup, data preparation and evaluation/run helpers only; individual plots are skipped.
WORKFLOW_CELLS = (2, 3, 5)


## Call the two dataset workflows

Only the setup, data and evaluation/experiment cells are executed, in separate namespaces. The combined notebook calls each source notebook's `run_experiment` for both sweeps and skips its individual plotting cells. Score normalization follows the source exactly: contiguity is $100[1-\overline H/\log(\min(K,10))]$ and delineation is $100(1-\mathrm{WCSS}/\mathrm{TSS})$ in standardized feature space. K-means uses 20 initializations and its fixed source seed.

In [2]:
def workflow_sources(notebook_path):
    notebook = json.loads(notebook_path.read_text(encoding='utf-8'))
    sources = [''.join(notebook['cells'][i]['source']) for i in WORKFLOW_CELLS]
    for source, expected in zip(sources, ('SEEDS =', 'StandardScaler()', 'def run_experiment(')):
        if expected not in source:
            raise ValueError(f'{notebook_path.name}: source layout changed; review workflow cells.')
    return sources


def validate_results(frame, dataset_id):
    title, _, reference_k = DATASETS[dataset_id]
    if len(frame) != 300 or not frame.dataset.eq(title).all() or not frame.K.eq(reference_k).all():
        raise ValueError(f'{title}: expected 300 rows at K={reference_k}.')
    if frame.duplicated(['experiment', 'value', 'seed', 'K']).any():
        raise ValueError(f'{title}: duplicate seed results.')
    if set(frame.experiment) != {'overlap', 'ramp'}:
        raise ValueError(f'{title}: missing sweep.')
    for experiment, values in [('overlap', OVERLAP_VALUES), ('ramp', RAMP_VALUES)]:
        part = frame[frame.experiment == experiment]
        if set(part.value) != set(values):
            raise ValueError(f'{title}: unexpected {experiment} values.')
        for value, group in part.groupby('value'):
            if set(group.seed) != set(SEEDS) or len(group) != len(SEEDS):
                raise ValueError(f'{title}: incomplete seeds at {experiment}={value}.')
        expected_f = part.value if experiment == 'overlap' else np.full(len(part), 0.5)
        expected_r = part.value if experiment == 'ramp' else np.full(len(part), 25)
        np.testing.assert_allclose(part.overlap_sc_factor, expected_f)
        np.testing.assert_allclose(part.phase3_ramp_iterations, expected_r)
    scores = frame[['spatial', 'multivariate']].to_numpy()
    if not np.isfinite(scores).all() or ((scores < -1e-8) | (scores > 100 + 1e-8)).any():
        raise ValueError(f'{title}: invalid scores.')
    baseline_f = frame[(frame.experiment == 'overlap') & (frame.value == 0.5)].sort_values('seed')
    baseline_r = frame[(frame.experiment == 'ramp') & (frame.value == 25)].sort_values('seed')
    np.testing.assert_allclose(baseline_f[['spatial', 'multivariate']],
                               baseline_r[['spatial', 'multivariate']], rtol=0, atol=0)


frames = []
for dataset_id, (title, filename, reference_k) in DATASETS.items():
    path = SOURCE_DIR / filename
    sources = workflow_sources(path)
    print(f'{title}: running both source sensitivity experiments', flush=True)
    namespace = {'__name__': '__sensitivity_workflow__'}
    for index, source in zip(WORKFLOW_CELLS, sources):
        exec(compile(source, f'{path}:cell{index}', 'exec'), namespace)
    if (tuple(namespace['SEEDS']) != SEEDS
        or tuple(namespace['OVERLAP_VALUES']) != OVERLAP_VALUES
        or tuple(namespace['RAMP_VALUES']) != RAMP_VALUES
        or namespace['N_NEIGHBORS'] != 10 or namespace['K_REFERENCE'] != reference_k):
        raise ValueError(f'{title}: source design changed; review the combined figure/table.')
    overlap = namespace['run_experiment']('overlap', OVERLAP_VALUES)
    ramp = namespace['run_experiment']('ramp', RAMP_VALUES)
    frame = pd.concat([overlap, ramp], ignore_index=True)
    frame.insert(0, 'dataset', title)
    validate_results(frame, dataset_id)
    assert len(namespace['run_cache']) == 270
    del namespace
    frames.append(frame)

sensitivity_results = pd.concat(frames, ignore_index=True)
print(f'{len(sensitivity_results)} rows; 30 paired seeds per parameter value and dataset.')


CMiO: running both source sensitivity experiments
CMiO: 1551 samples, 35 features; neighborhood k=10; cluster counts=(7,); 30 seeds


,Samples
True class,
0,22
1,364
2,306
3,111
4,252
5,216
6,280


overlap: completed seed 1/30
overlap: completed seed 2/30
overlap: completed seed 3/30
overlap: completed seed 4/30
overlap: completed seed 5/30
overlap: completed seed 6/30
overlap: completed seed 7/30
overlap: completed seed 8/30
overlap: completed seed 9/30
overlap: completed seed 10/30
overlap: completed seed 11/30
overlap: completed seed 12/30
overlap: completed seed 13/30
overlap: completed seed 14/30
overlap: completed seed 15/30
overlap: completed seed 16/30
overlap: completed seed 17/30
overlap: completed seed 18/30
overlap: completed seed 19/30
overlap: completed seed 20/30
overlap: completed seed 21/30
overlap: completed seed 22/30
overlap: completed seed 23/30
overlap: completed seed 24/30
overlap: completed seed 25/30
overlap: completed seed 26/30
overlap: completed seed 27/30
overlap: completed seed 28/30
overlap: completed seed 29/30
overlap: completed seed 30/30
ramp: completed seed 1/30
ramp: completed seed 2/30
ramp: completed seed 3/30
ramp: completed seed 4/30
ramp:

,Samples
True class,
0,27
1,32
2,24
3,47
4,28
5,201
6,20
7,33
8,299


overlap: completed seed 1/30
overlap: completed seed 2/30
overlap: completed seed 3/30
overlap: completed seed 4/30
overlap: completed seed 5/30
overlap: completed seed 6/30
overlap: completed seed 7/30
overlap: completed seed 8/30
overlap: completed seed 9/30
overlap: completed seed 10/30
overlap: completed seed 11/30
overlap: completed seed 12/30
overlap: completed seed 13/30
overlap: completed seed 14/30
overlap: completed seed 15/30
overlap: completed seed 16/30
overlap: completed seed 17/30
overlap: completed seed 18/30
overlap: completed seed 19/30
overlap: completed seed 20/30
overlap: completed seed 21/30
overlap: completed seed 22/30
overlap: completed seed 23/30
overlap: completed seed 24/30
overlap: completed seed 25/30
overlap: completed seed 26/30
overlap: completed seed 27/30
overlap: completed seed 28/30
overlap: completed seed 29/30
overlap: completed seed 30/30
ramp: completed seed 1/30
ramp: completed seed 2/30
ramp: completed seed 3/30
ramp: completed seed 4/30
ramp:

## Supplementary table: mean contiguity and delineation at every f and R

Statistics use unrounded seed scores; SD is the sample standard deviation ($ddof=1$). Both fixed and varied parameter values are shown. The shared baseline appears in both sweeps, with identical scores. Values quantify algorithm variation on fixed datasets.

In [3]:
summary = (sensitivity_results.groupby(
    ['dataset', 'experiment', 'value', 'overlap_sc_factor', 'phase3_ramp_iterations', 'K'],
    sort=False, as_index=False)
    .agg(n_seeds=('seed', 'count'),
         contiguity_mean=('spatial', 'mean'), contiguity_sd=('spatial', 'std'),
         delineation_mean=('multivariate', 'mean'), delineation_sd=('multivariate', 'std')))
assert len(summary) == 20 and summary.n_seeds.eq(30).all()
assert np.isfinite(summary[['contiguity_mean', 'contiguity_sd',
                           'delineation_mean', 'delineation_sd']]).all().all()
supplementary_table = summary[['dataset', 'experiment', 'overlap_sc_factor',
                              'phase3_ramp_iterations', 'K', 'n_seeds']].copy()
supplementary_table.columns = ['Dataset', 'Sweep', 'f', 'R (iterations)', 'K', 'Seeds']
supplementary_table['Sweep'] = supplementary_table.Sweep.map({'overlap': 'Overlap f', 'ramp': 'Ramp R'})
for metric, label in [('contiguity', 'Spatial contiguity (%)'),
                      ('delineation', 'Multivariate delineation (%)')]:
    supplementary_table[label] = [f'{mean:.2f} ± {sd:.2f}' for mean, sd in
        zip(summary[f'{metric}_mean'], summary[f'{metric}_sd'])]
caption = ('SC-PaCMAP sensitivity for CMiO and North Batten. Mean ± sample SD across '
           '30 paired algorithm seeds; fitting and evaluation k=10. The f sweep fixes R=25; '
           'the R sweep fixes f=0.5. K=7 for CMiO and K=15 for North Batten.')
style = (supplementary_table.style.hide(axis='index').set_caption(caption)
         .format({'f': '{:.2f}', 'R (iterations)': '{:.0f}'})
         .set_properties(**{'font-family': 'Arial, sans-serif', 'font-size': '10pt',
                            'padding': '4px 8px'})
         .set_table_styles([{'selector': 'caption', 'props': [('text-align', 'left')]},
                            {'selector': 'th', 'props': [('text-align', 'left')]}]))
display(HTML(style.to_html()))


Dataset,Sweep,f,R (iterations),K,Seeds,Spatial contiguity (%),Multivariate delineation (%)
CMiO,Overlap f,0.00,25,7,30,95.08 ± 1.43,37.48 ± 1.93
CMiO,Overlap f,0.25,25,7,30,93.37 ± 0.92,39.20 ± 0.32
CMiO,Overlap f,0.50,25,7,30,91.32 ± 0.87,39.51 ± 0.25
CMiO,Overlap f,0.75,25,7,30,87.05 ± 1.54,40.99 ± 1.25
CMiO,Overlap f,1.00,25,7,30,82.33 ± 0.79,41.72 ± 1.35
CMiO,Ramp R,0.50,0,7,30,91.37 ± 1.14,38.51 ± 1.89
CMiO,Ramp R,0.50,10,7,30,90.93 ± 1.07,39.12 ± 1.39
CMiO,Ramp R,0.50,25,7,30,91.32 ± 0.87,39.51 ± 0.25
CMiO,Ramp R,0.50,50,7,30,91.37 ± 1.02,39.60 ± 0.31
CMiO,Ramp R,0.50,100,7,30,91.47 ± 1.05,39.69 ± 0.44


## Compact 2 x 2 sensitivity figure for Results

Rows: overlap factor $f$ and ramp duration $R$. Columns: spatial contiguity and multivariate delineation. Both datasets appear in every panel; markers show means, error bars show ±1 sample SD. Horizontal positions are the actual parameter values. Dashed vertical lines mark the default values.

In [4]:
COLORS = {
    'CMiO': '#0072B2',
    'North Batten': '#D55E00'
}

MARKERS = {
    'CMiO': 'o',
    'North Batten': 's'
}

with plt.rc_context({
    'figure.dpi': 600,
    'font.size': 8,
    'axes.titlesize': 9,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'axes.spines.top': False,
    'axes.spines.right': False
}):

    fig, axes = plt.subplots(
        2, 2,
        figsize=(7.1, 4.8),
        layout='constrained'
    )

    for row, (experiment, values, default, xlabel) in enumerate([
        (
            'overlap',
            OVERLAP_VALUES,
            0.5,
            'Overlap factor, $f$ ($R=25$)'
        ),
        (
            'ramp',
            RAMP_VALUES,
            25,
            'Ramp duration, $R$ (iterations; $f=0.5$)'
        ),
    ]):

        for col, (metric, title) in enumerate([
            ('contiguity', 'Spatial contiguity'),
            ('delineation', 'Multivariate delineation')
        ]):

            ax = axes[row, col]

            # Default parameter value
            ax.axvline(
                default,
                color='0.35',
                linestyle='--',
                linewidth=1.0,
                zorder=1
            )

            # Dataset curves
            for dataset in COLORS:

                part = summary[
                    (summary.dataset == dataset) &
                    (summary.experiment == experiment)
                ].sort_values('value')

                ax.errorbar(
                    part.value,
                    part[f'{metric}_mean'],
                    yerr=part[f'{metric}_sd'],
                    color=COLORS[dataset],
                    marker=MARKERS[dataset],
                    markersize=3.5,
                    linewidth=1,
                    elinewidth=0.8,
                    capsize=2,
                    label=dataset
                )

            # Axis labels and ticks
            ax.set_xticks(values)
            ax.set_xlabel(xlabel)

            # Show y-axis label on ALL four panels
            ax.set_ylabel(f'{title} (%)')

            # Panel labels
            ax.set_title(
                f'({chr(97 + 2 * row + col)})',
                loc='left'
            )

            # Grid
            ax.grid(False, axis='x', which='both')
            ax.grid(
                True,
                axis='y',
                which='major',
                alpha=0.25
            )

            # Use one shared score scale for each metric
            # across the two sensitivity sweeps.
            lo = (
                summary[f'{metric}_mean'] -
                summary[f'{metric}_sd']
            ).min()

            hi = (
                summary[f'{metric}_mean'] +
                summary[f'{metric}_sd']
            ).max()

            padding = max(
                1.0,
                0.12 * (hi - lo)
            )

            ax.set_ylim(
                max(0, lo - padding),
                min(100, hi + padding)
            )

    # Shared legend
    handles = [
        Line2D(
            [],
            [],
            color=COLORS[name],
            marker=MARKERS[name],
            markersize=4,
            linewidth=1,
            label=name
        )
        for name in COLORS
    ]

    fig.legend(
        handles=handles,
        loc='outside upper center',
        ncol=2,
        frameon=False
    )

    plt.show()


figure_caption = (
    'SC-PaCMAP sensitivity to overlap factor f (a,b; R=25) and Phase-3 ramp duration R '
    '(c,d; f=0.5) in CMiO and North Batten. Columns show spatial contiguity (a,c) and '
    'multivariate delineation (b,d), both as percentages with higher scores indicating '
    'better performance. Points show means and error bars ±1 sample SD across 30 paired '
    'algorithm seeds on each fixed dataset. Fitting and evaluation neighbourhoods are k=10; '
    'K-means uses K=7 (CMiO) or K=15 (North Batten). Dashed lines mark defaults f=0.5 '
    'and R=25. The shared baseline is reused across sweeps. Full mean±SD results are '
    'reported in the supplementary sensitivity table.'
)

print(figure_caption)


SC-PaCMAP sensitivity to overlap factor f (a,b; R=25) and Phase-3 ramp duration R (c,d; f=0.5) in CMiO and North Batten. Columns show spatial contiguity (a,c) and multivariate delineation (b,d), both as percentages with higher scores indicating better performance. Points show means and error bars ±1 sample SD across 30 paired algorithm seeds on each fixed dataset. Fitting and evaluation neighbourhoods are k=10; K-means uses K=7 (CMiO) or K=15 (North Batten). Dashed lines mark defaults f=0.5 and R=25. The shared baseline is reused across sweeps. Full mean±SD results are reported in the supplementary sensitivity table.
